# Analyzing River Thames Water Levels
Time series data is everywhere, from watching your stock portfolio to monitoring climate change, and even live-tracking as local cases of a virus become a global pandemic. In this project, you’ll work with a time series that tracks the tide levels of the Thames River. You’ll first load the data and inspect it data visually, and then perform calculations on the dataset to generate some summary statistics. You’ll end by reducing the time series to its component attributes and analyzing them. 

The original dataset is available from the British Oceanographic Data Center.

Here's a map of the locations of the tidal meters along the River Thames in London.

![](locations.png)

The provided datasets are in the `data` folder in this workspace. For this project, you will work with one of these files, `10-11_London_Bridge.txt`, which contains comma separated values for water levels in the Thames River at the London Bridge. After you've finished the project, you can use your same code to analyze data from the other files (at other spots in the UK where tidal data is collected) if you'd like. 

The TXT file contains data for three variables, described in the table below. 

| Variable Name | Description | Format |
| ------------- | ----------- | ------ |
| Date and time | Date and time of measurement to GMT. Note the tide gauge is accurate to one minute. | dd/mm/yyyy hh:mm:ss |
| Water level | High or low water level measured by tide meter. Tide gauges are accurate to 1 centimetre. | metres (Admiralty Chart Datum (CD), Ordnance Datum Newlyn (ODN or Trinity High Water (THW)) | 
| Flag | High water flag = 1, low water flag = 0 | Categorical (0 or 1) |



### Analyze Thames River tidal data to track changes in high-tide and low-tide frequency over time.

### The data is in the data/10-11_London_Bridge.txt file.

#### 1. Find the mean, median, and interquartile range for high- and low-tide data and save them as two separate pandas Series.

#### 2. Calculate the annual percentage of days with very high tide levels (90th percentile of high tide days) and low-tide days (below the 10th percentile), and store the results for each year as floats in two two-column DataFrames with the index reset.

#### 3. Create a dictionary named solution with a summary of your data analysis, with these key-value pairs:

#### {high_statistics: high-tide stats, low_statistics: low-tide stats, very_high_ratio: high-tide ratio data, very_low_ratio: low-tide ratio data}

In [131]:
# We've imported your first Python package for you, along with a function you will need called IQR
import pandas as pd  
import numpy as np             

def IQR(column): 
    """ Calculates the interquartile range (IQR) for a given DataFrame column using the quantile method """
    q25, q75 = column.quantile([0.25, 0.75])
    return q75-q25

# Begin coding here ...

### Question 1

In [132]:
df = pd.read_csv('data/10-11_London_Bridge.txt', sep = ',')
df[" water level (m ODN)"] = pd.to_numeric(df[" water level (m ODN)"], errors='coerce')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115503 entries, 0 to 115502
Data columns (total 4 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Date and time         115503 non-null  object 
 1    water level (m ODN)  115489 non-null  float64
 2    flag                 115503 non-null  int64  
 3    HW=1 or LW=0         0 non-null       float64
dtypes: float64(2), int64(1), object(1)
memory usage: 3.5+ MB
None


In [133]:
high_tide_filt = df[df[' flag'] == 1]
high = high_tide_filt.groupby(' flag')[' water level (m ODN)'].agg(mean = 'mean', median = 'median', iqr = IQR)
print(high)

           mean  median     iqr
 flag                          
1      3.318373  3.3526  0.7436


In [134]:
low_tide_filt = df[df[' flag'] == 0]
low = low_tide_filt.groupby(' flag')[' water level (m ODN)'].agg(mean = 'mean', median = 'median', iqr = IQR)
print(low)

           mean  median     iqr
 flag                          
0     -2.383737 -2.4129  0.5382


### Question 2

In [135]:
df['year'] = pd.to_datetime(df['Date and time'], format="%d/%m/%Y %H:%M:%S").dt.year
high_thresh = df[' water level (m ODN)'].quantile(0.9)
low_thresh = df[' water level (m ODN)'].quantile(0.1)

In [136]:
df['very_high'] = df[' water level (m ODN)'] > high_thresh
df['very_low'] = df[' water level (m ODN)'] < low_thresh

print(df)

              Date and time   water level (m ODN)   flag   HW=1 or LW=0  year  \
0       01/05/1911 15:40:00                3.7130      1            NaN  1911   
1       02/05/1911 11:25:00               -2.9415      0            NaN  1911   
2       02/05/1911 16:05:00                3.3828      1            NaN  1911   
3       03/05/1911 11:50:00               -2.6367      0            NaN  1911   
4       03/05/1911 16:55:00                2.9256      1            NaN  1911   
...                     ...                   ...    ...            ...   ...   
115498  30/12/1995 20:44:00                3.2900      1            NaN  1995   
115499  31/12/1995 02:32:00               -1.6000      0            NaN  1995   
115500  31/12/1995 08:59:00                3.2000      1            NaN  1995   
115501  31/12/1995 15:03:00               -1.8000      0            NaN  1995   
115502  31/12/1995 21:50:00                3.1700      1            NaN  1995   

        very_high  very_low

In [137]:
high_tides = (df.groupby('year')['very_high'].mean() * 100).reset_index(name='very_high_pct')

print(high_tides)

    year  very_high_pct
0   1911       0.631579
1   1912       4.766187
2   1913       7.244212
3   1914       5.724638
4   1915       8.459215
..   ...            ...
80  1991      11.268604
81  1992      11.456860
82  1993      14.071429
83  1994      16.028369
84  1995      14.326241

[85 rows x 2 columns]


In [138]:
low_tides = (df.groupby('year')['very_low'].mean() * 100).reset_index(name='very_low_pct')

print(low_tides)

    year  very_low_pct
0   1911      7.789474
1   1912      8.093525
2   1913      3.435400
3   1914      5.507246
4   1915      4.984894
..   ...           ...
80  1991     12.686038
81  1992     10.749646
82  1993     10.214286
83  1994     10.070922
84  1995     10.070922

[85 rows x 2 columns]


In [139]:
solution = {'high_statistics': 'high-tidestats', 'low_statistics': 'low-tide stats', 'very_high_ratio': 'high-tide ratio data', 'very_low_ratio': 'low-tide ratio data'}

In [140]:
high_stats = high.loc[1]
solution['high_statistics'] = high_stats.to_dict()

low_stats = low.loc[0]
solution['low_statistics'] = low_stats.to_dict()
print(solution)

{'high_statistics': {'mean': 3.3183726156555977, 'median': 3.3526, 'iqr': 0.7436000000000003}, 'low_statistics': {'mean': -2.3837365821465784, 'median': -2.4129, 'iqr': 0.5382000000000002}, 'very_high_ratio': 'high-tide ratio data', 'very_low_ratio': 'low-tide ratio data'}


In [141]:
very_high_dict = dict(zip(high_tides["year"], high_tides["very_high_pct"]))
very_low_dict = dict(zip(low_tides["year"], low_tides["very_low_pct"]))

solution["very_high_ratio"] = very_high_dict
solution["very_low_ratio"] = very_low_dict

print(solution)

{'high_statistics': {'mean': 3.3183726156555977, 'median': 3.3526, 'iqr': 0.7436000000000003}, 'low_statistics': {'mean': -2.3837365821465784, 'median': -2.4129, 'iqr': 0.5382000000000002}, 'very_high_ratio': {1911: 0.631578947368421, 1912: 4.766187050359712, 1913: 7.2442120985810305, 1914: 5.72463768115942, 1915: 8.459214501510575, 1916: 10.12472487160675, 1917: 7.340116279069768, 1918: 5.88235294117647, 1919: 8.130081300813007, 1920: 7.261247040252565, 1921: 4.901185770750988, 1922: 6.908831908831909, 1923: 6.678230702515179, 1924: 7.675111773472429, 1925: 7.016300496102056, 1926: 7.548526240115025, 1927: 7.780157030692362, 1928: 7.834757834757834, 1929: 5.987170349251604, 1930: 6.325515280739161, 1931: 5.844618674269423, 1932: 5.374823196605375, 1933: 5.464868701206529, 1934: 2.7659574468085104, 1935: 6.89410092395167, 1936: 8.002832861189802, 1937: 9.737029140014215, 1938: 7.097232079488999, 1939: 10.149041873669269, 1940: 7.44153082919915, 1941: 8.169934640522875, 1942: 7.00500357